# Cell 1: Exploring

In [ ]:
import os, shutil, itertools
import psutil

def peek(path, depth=0, max_depth=3, limit=15):
    try:
        entries = list(itertools.islice(os.scandir(path), limit + 1))
    except (PermissionError, FileNotFoundError) as err:
        print("  " * depth + f"[cannot read: {err}]")
        return
    for e in sorted(entries[:limit], key=lambda x: x.name):
        pad = "  " * depth
        if e.is_dir():
            print(pad + e.name + "/")
            if depth < max_depth:
                peek(e.path, depth + 1, max_depth, limit)
        else:
            print(pad + f"{e.name}  ({e.stat().st_size / 1e6:.2f} MB)")
    if len(entries) > limit:
        print("  " * depth + "... more entries not shown")

peek("/kaggle/input")

print()
for d in ["/kaggle/input", "/kaggle/working", "/kaggle/temp"]:
    if os.path.exists(d):
        total, used, free = shutil.disk_usage(d)
        print(f"{d}: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB, CPUs: {os.cpu_count()}")

try:
    import pydicom
    print("pydicom", pydicom.__version__)
except ImportError:
    print("pydicom not installed")

# Cell 2: CSV schemas

In [ ]:
import os
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

dfs = {}
for name in ["train", "train_series", "test", "test_series", "sample_submission"]:
    df = pd.read_csv(ROOT / f"{name}.csv")
    dfs[name] = df
    print(f"\n=== {name}.csv  shape={df.shape}")
    for col in df.columns:
        s = df[col]
        ex = str(s.dropna().iloc[0])[:80] if s.notna().any() else "ALL NULL"
        print(f"  {col:<35} {str(s.dtype):<8} nulls={s.isna().sum():<6} "
              f"unique={s.nunique():<7} e.g. {ex!r}")

# Cell 3: inside the image folders

In [ ]:
def count_entries(p):
    with os.scandir(p) as it:
        return sum(1 for _ in it)

print("train_series folders:", count_entries(ROOT / "train_series"))
print("test_series folders:", count_entries(ROOT / "test_series"))

first = next(os.scandir(ROOT / "train_series"))
print("\nsample folder:", first.name)
entries = sorted(os.scandir(first.path), key=lambda e: e.name)
print("entries:", len(entries))
for e in entries[:8]:
    kind = "dir " if e.is_dir() else "file"
    size = 0 if e.is_dir() else e.stat().st_size / 1e3
    print(f"  {kind} {e.name}  {size:.1f} KB")

# Cell 4: labels and reports

In [ ]:
LABELS = [c for c in dfs["sample_submission"].columns if c != "StudyInstanceUID"]
train = dfs["train"]
ser = dfs["train_series"]

has = train[LABELS].notna()
print("rows with any label:", has.any(axis=1).sum(), " rows with all labels:", has.all(axis=1).sum())

gold = train[has.all(axis=1)]
print(f"\ngold positives (n={len(gold)}):")
print(gold[LABELS].sum().astype(int).to_string())

print("\nreport length (chars):", train["Report"].str.len().describe().round(0).to_dict())

dup = train[train["Report"].duplicated(keep=False)]
print("\nstudies sharing a report:", len(dup), " distinct shared texts:", dup["Report"].nunique())
for text, n in dup["Report"].value_counts().head(5).items():
    print(f"\n[{n}x] {text[:300]!r}")

for _, r in gold.head(2).iterrows():
    print("\n" + "-" * 60)
    print(r["Report"])
    print({k: int(r[k]) for k in LABELS})

# Cell 5: series composition

In [ ]:
print("series per study:", ser.groupby("StudyInstanceUID").size().describe().round(2).to_dict())
print()
print(ser.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).size().to_string())

planes = ser.groupby("StudyInstanceUID")["Anatomical_Plane"].agg(set)
print()
for p in ["Sagittal", "Coronal", "Axial"]:
    print(f"studies with no {p}:", (~planes.apply(lambda s: p in s)).sum())

# Cell 6: DICOM size, headers, decode speed

In [ ]:
import time
import pydicom

sample = ser.sample(20, random_state=0)
rows = []
for r in sample.itertuples():
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    files = list(os.scandir(d))
    rows.append((r.Anatomical_Plane, len(files), sum(f.stat().st_size for f in files) / 1e6))
s = pd.DataFrame(rows, columns=["plane", "n_files", "mb"])
print(s.to_string())
print(f"\nmean files/series {s.n_files.mean():.1f}, mean MB/series {s.mb.mean():.1f}")
print(f"rough total: {s.mb.mean() * len(ser) / 1e3:.0f} GB")

r = sample.iloc[0]
d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
paths = sorted(e.path for e in os.scandir(d))
ds = pydicom.dcmread(paths[0], stop_before_pixels=True)
print()
for tag in ["Modality", "SeriesDescription", "ImageOrientationPatient", "ImagePositionPatient",
            "PixelSpacing", "SliceThickness", "SpacingBetweenSlices", "Rows", "Columns",
            "InstanceNumber", "MagneticFieldStrength", "Manufacturer", "RepetitionTime",
            "EchoTime", "BitsStored", "PhotometricInterpretation", "PatientID", "InstitutionName"]:
    print(f"  {tag:<26} {ds.get(tag, 'MISSING')}")
print("  transfer syntax:", ds.file_meta.TransferSyntaxUID.name)

t = time.time()
arrs = [pydicom.dcmread(p).pixel_array for p in paths]
print(f"\ndecoded {len(arrs)} slices in {time.time() - t:.2f}s, shape {arrs[0].shape}, dtype {arrs[0].dtype}")

# Cell 7: report languages

In [ ]:
!pip install -q langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang_of(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"

train["lang"] = train["Report"].map(lang_of)
train["is_gold"] = train["StudyInstanceUID"].isin(set(gold["StudyInstanceUID"]))
print(pd.crosstab(train["lang"], train["is_gold"], margins=True)
        .sort_values("All", ascending=False).to_string())

# Cell 8: header scan, pilot on 200 series

In [ ]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

N = len(ser)

def plane_from_iop(iop):
    v = np.array(iop, dtype=float)
    n = np.cross(v[:3], v[3:])
    return ["Sagittal", "Coronal", "Axial"][int(np.argmax(np.abs(n)))]

def scan_series(row):
    out = {"StudyInstanceUID": row.StudyInstanceUID,
           "SeriesInstanceUID": row.SeriesInstanceUID, "error": None}
    try:
        d = ROOT / "train_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        entries = sorted(os.scandir(d), key=lambda e: e.name)
        out["n_files"] = len(entries)
        out["mb"] = sum(e.stat().st_size for e in entries) / 1e6
        ds = pydicom.dcmread(entries[len(entries) // 2].path, stop_before_pixels=True)
        out["desc"] = ds.get("SeriesDescription")
        out["manufacturer"] = ds.get("Manufacturer")
        out["PatientID"] = ds.get("PatientID")
        out["rows"], out["cols"] = ds.get("Rows"), ds.get("Columns")
        ps = ds.get("PixelSpacing")
        out["px_mm"] = float(ps[0]) if ps else None
        out["thick_mm"] = float(ds.SliceThickness) if "SliceThickness" in ds else None
        iop = ds.get("ImageOrientationPatient")
        out["iop_plane"] = plane_from_iop(iop) if iop else None
    except Exception as err:
        out["error"] = repr(err)[:200]
    return out

t = time.time()
with ThreadPoolExecutor(16) as ex:
    recs = list(ex.map(scan_series, ser.head(N).itertuples(index=False)))
el = time.time() - t
inv = pd.DataFrame(recs)
print(f"{N} series in {el:.1f}s -> full estimate {el / N * len(ser) / 60:.1f} min")
print("errors:", inv["error"].notna().sum())
print(inv.drop(columns=["StudyInstanceUID", "SeriesInstanceUID"]).head(10).to_string())

# Cell 9: summary and first artifact

In [ ]:
inv = inv.merge(ser, on=["StudyInstanceUID", "SeriesInstanceUID"])
print("errors:", inv["error"].notna().sum())
print("plane agreement:", round((inv["iop_plane"] == inv["Anatomical_Plane"]).mean(), 4))
print(pd.crosstab(inv["Anatomical_Plane"], inv["iop_plane"]))

pid = inv.groupby("PatientID")["StudyInstanceUID"].nunique()
print("\npatients:", len(pid), " with >1 study:", (pid > 1).sum())
print("total GB:", round(inv["mb"].sum() / 1e3, 1))
print("n_files:", inv["n_files"].describe().round(1).to_dict())
print("\n" + inv["manufacturer"].value_counts().to_string())

fs = inv[inv["Fluid_Sensitive"] == 1].groupby("StudyInstanceUID")["Anatomical_Plane"].nunique()
print("\nstudies with FS in all 3 planes:", (fs == 3).sum(), "of", ser["StudyInstanceUID"].nunique())

inv.to_csv("/kaggle/working/series_inventory.csv", index=False)
train[["StudyInstanceUID", "lang", "is_gold"]].to_csv("/kaggle/working/study_inventory.csv", index=False)

# Cell 10: site proxy and FS series per plane

In [ ]:
def vendor(m):
    m = str(m).lower()
    for v in ["siemens", "ge", "philips", "toshiba", "canon", "hitachi"]:
        if m.startswith(v):
            return v
    return "other"

inv["vendor"] = inv["manufacturer"].map(vendor)
study_vendor = inv.groupby("StudyInstanceUID")["vendor"].agg(lambda s: s.mode()[0])
st = train.set_index("StudyInstanceUID")[["lang", "is_gold"]].join(study_vendor)
print(pd.crosstab(st["lang"], st["vendor"], margins=True).to_string())

planes = ["Sagittal", "Coronal", "Axial"]
fs = inv[inv["Fluid_Sensitive"] == 1]
cnt = (fs.groupby(["StudyInstanceUID", "Anatomical_Plane"]).size().unstack(fill_value=0)
         .reindex(index=train["StudyInstanceUID"], columns=planes, fill_value=0))
print("\nFS series per study, by plane:")
for p in planes:
    print(f"  {p:<9}", cnt[p].value_counts().sort_index().to_dict())
print("studies with FS in all 3 planes:", (cnt > 0).all(axis=1).sum())

cols = ["n_files", "rows", "px_mm", "thick_mm"]
print("\n" + fs.groupby("Anatomical_Plane")[cols].agg(["median", "min", "max"]).round(2).to_string())

st.reset_index().to_csv("/kaggle/working/study_inventory.csv", index=False)
inv.to_csv("/kaggle/working/series_inventory.csv", index=False)

# Cell 11: are the gold studies representative?

In [ ]:
g = st[st["is_gold"]]
print(pd.crosstab(g["lang"], g["vendor"], margins=True).to_string())
print("\ngold with FS in all 3 planes:", (cnt.loc[g.index] > 0).all(axis=1).sum(), "of", len(g))